# AgentComet - Complete Feature Guide

This notebook demonstrates **all** features of AgentComet:

1. **Agent SDK** — Build agents with `@tool`, `Agent` class, and `create_agent`
2. **Tool System** — `@tool` decorator, `ToolSpec`, builtin & custom tools
3. **UAF Export & Load** — Export agents to `.uaf` and reload them
4. **UAFAgent (Legacy)** — Load and run pre-built `.uaf` agents
5. **Memory Management** — Conversation history with full/limited modes
6. **State Persistence** — Embedded state + versioned rollback
7. **LLM Integration** — Use built-in providers
8. **Orchestration** — Connect multiple agents
9. **Workflow Templates** — Pipeline, Fan-Out/Fan-In, Map-Reduce

**Prerequisites:**
```bash
pip install uaf_compiler pyyaml requests
pip install -e .  # Install agentcomet
```

---

## Setup — LLM Instance

All examples use a single `Ollama` instance. No `.as_langchain()` needed for SDK agents.

In [ ]:
import os
from agentcomet.models import Ollama

# One LLM instance for all examples
llm = Ollama(model="gemma3:4b")
print("LLM ready:", llm.model)

---

## 1. Agent SDK — The Core of AgentComet

AgentComet provides a clean, declarative SDK to build agents. No LangChain required.

### 1A. Custom Tools with `@tool` Decorator

The `@tool` decorator converts any Python function into a `ToolSpec` — with auto-extracted name, description, and JSON schema from type hints.

In [ ]:
from agentcomet.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiplies two numbers together."""
    return a * b

@tool
def calculator(expression: str) -> str:
    """Evaluates a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

print("Tool Name:", multiply.name)
print("Description:", multiply.description)
print("Schema:", multiply.schema)
print()
print("Calculator Schema:", calculator.schema)

### 1B. Builtin Tools

AgentComet ships with builtin tools like `read` and `write` for file operations.

In [ ]:
from agentcomet.tools import read, write

print("Builtin Tool:", read.name, "-", read.description)
print("Builtin Tool:", write.name, "-", write.description)

### 1C. Declarative Agent with `create_agent`

The simplest way to make an agent — no subclassing needed.

In [ ]:
from agentcomet import create_agent

agent = create_agent(
    name="math-bot",
    description="A simple math assistant",
    llm=llm,
    tools=[multiply, calculator],
    memory=True
)

print(agent.run("What is 12 times 8?"))

### 1D. Custom Agent Class (Recommended for Complex Agents)

For full control, subclass `Agent` and define everything in `setup()`.  
`name`, `description`, and `author` can be set inside `setup()`.  
Configuration not defined in `setup()` (e.g. `llm`) can be passed at instantiation time.

In [ ]:
from agentcomet import Agent

class MyMathAgent(Agent):
    def setup(self):
        self.name = "math-bot-custom"
        self.description = "Custom math assistant with tool calling"
        self.author = "Vaibhav"
        self.use_memory(True)
        self.add_tools(multiply, calculator)

    def run(self, input: str):
        return self.chat(f"Solve this: {input}")

# LLM is provided at instantiation time
agent = MyMathAgent(llm=llm)
print(agent.run("What is 6 times 7?"))

### 1E. Override Defaults at Instantiation

Constructor kwargs always override `setup()` values. This lets you reuse the same agent class with different configurations.

In [ ]:
# Same agent class, different name
agent_v2 = MyMathAgent(
    name="math-bot-v2",
    llm=llm,
    description="Version 2 of the math assistant"
)

print("Agent name:", agent_v2.name)               # math-bot-v2 (overridden)
print("Description:", agent_v2.description)        # Version 2 (overridden)
print("Author:", agent_v2.author)                  # Vaibhav (from setup)

### 1F. Default Values

If you don't set `name`, `description`, or `author`, these defaults apply:

| Field | Default |
|---|---|
| `name` | `"default-agent"` |
| `description` | `"This is a tool calling agent that solves tasks using its assigned tools."` |
| `author` | `"AgentComet"` |

In [ ]:
class BareAgent(Agent):
    def setup(self):
        self.add_tools(multiply)

bare = BareAgent()
print("Name:", bare.name)
print("Description:", bare.description)
print("Author:", bare.author)

---

## 2. UAF Export & Load

Agents built with AgentComet SDK can be exported to `.uaf` files (portable archives)  
and reloaded using `load_agent()` — which auto-routes to the correct runtime based on the `sdk.name` in `agent.yaml`.

In [ ]:
from agentcomet import load_agent

# Export agent to .uaf
agent.export("my_math_agent.uaf")

# Load it back — auto-detects AgentComet SDK from agent.yaml
loaded = load_agent("my_math_agent.uaf")
print("Loaded type:", type(loaded))
print(loaded.run("What is 100 / 4?"))

# Cleanup
os.remove("my_math_agent.uaf")

### UAF Archive Structure

The exported `.uaf` file is a `tar.gz` containing:

```
my-agent.uaf
├── agent.yaml           # V2 Manifest (sdk: agentcomet)
├── agent.py             # Auto-generated runner
├── tools.py             # Custom tools (if any)
├── agent.state          # Memory state (if enabled)
├── requirements.txt
└── sdk/
    └── agentcomet.json   # SDK metadata
```

No LangChain references. No internal engine leakage. Clean and portable.

---

## 3. UAFAgent — Load Pre-Built UAF Agents (Legacy)

For agents built with external frameworks (LangGraph, CrewAI, etc.), use `UAFAgent`.  
This path requires `ollama.as_langchain()` to provide a LangChain-compatible LLM.

In [ ]:
from agentcomet.agents import UAFAgent

AGENT_PATH = "dummy_agents/mathematics-reasoning-agent.uaf"

# Legacy path needs LangChain-compatible LLM
langchain_llm = llm.as_langchain()

agent = UAFAgent("math_agent", AGENT_PATH, llm=langchain_llm, restore_state=False)

response = agent.invoke("What is 25 * 4?")
print("Content:", response.content)

agent.cleanup()

---

## 4. Memory Management

Agents can maintain conversation history for multi-turn interactions.

### Memory Modes:
- `memory=None` (default) — No memory, each call is independent
- `memory="full"` — Keep all messages forever
- `memory=N` — Keep only the last N messages (sliding window)

In [ ]:
agent = UAFAgent("conversation", AGENT_PATH, llm=langchain_llm, memory="full", restore_state=False)

r1 = agent.invoke("What is 10 + 5?")
print("Q1:", r1.content)

r2 = agent.invoke("Double that result")
print("Q2:", r2.content)

r3 = agent.invoke("Now subtract 10")
print("Q3:", r3.content)

print(f"\nTotal messages in history: {len(agent.get_history())}")

### Memory Methods

In [ ]:
history = agent.get_history()
print(f"History length: {len(history)}")
for i, msg in enumerate(history):
    print(f"  [{i}] {type(msg).__name__}: {str(msg.content)[:50]}...")

agent.clear_memory()
print(f"\nAfter clear: {len(agent.get_history())} messages")

agent.cleanup()

---

## 5. State Persistence

Two ways to persist agent state:

### A. Embedded State (inside UAF)
- `save_agent(path)` — Embed current memory inside a new .uaf file
- `restore_state=True` (DEFAULT) — Auto-load embedded state

### B. External Versioned State (.agentcomet/states/)
- `save_state()` — Save versioned checkpoint, returns hash
- `load_state(hash)` — Switch between versions / rollback
- `show_states()` — View all saved versions
- `delete_state(hash)` — Remove a version

In [ ]:
agent = UAFAgent("stateful_agent", AGENT_PATH, llm=langchain_llm, memory="full", restore_state=False)

agent.invoke("My name is Alice")
agent.invoke("I'm learning about AI")
agent.invoke("Remember: the secret code is 42")

print(f"Current memory: {len(agent.get_history())} messages")

hash1 = agent.save_state()
print(f"Saved checkpoint: {hash1}")

agent.save_agent("agent_with_memory.uaf")

In [ ]:
restored_agent = UAFAgent("restored", "agent_with_memory.uaf", llm=langchain_llm, memory="full")

print(f"Restored {len(restored_agent.get_history())} messages from embedded state")

response = restored_agent.invoke("What's the secret code?")
print("Agent remembers:", response.content)

restored_agent.cleanup()

if os.path.exists('agent_with_memory.uaf'):
    os.remove('agent_with_memory.uaf')

---

## 6. LLM Providers

AgentComet includes built-in LLM providers. Pass them directly to SDK agents.

In [ ]:
from agentcomet.models import Ollama, OpenAIChat, Gemini, Anthropic, OpenRouter, Perplexity

# Ollama (local) — works directly with Agent SDK
ollama = Ollama(model="gemma3:4b")

# Direct usage
result = ollama.generate("What is the capital of France?")
print("Direct call:", result[:100])

# For legacy UAFAgent, use .as_langchain()
# langchain_llm = ollama.as_langchain()

# Other providers (require API keys)
# openai = OpenAIChat(model="gpt-4o")
# gemini = Gemini(model="gemini-1.5-flash")
# claude = Anthropic(model="claude-3-5-sonnet")

---

## 7. Orchestration — Connecting Multiple Agents

Use `AgentOrchestrator` to build and run multi-agent workflows.

In [ ]:
from agentcomet.orchestrators import AgentOrchestrator
from langchain_core.messages import HumanMessage

orch = AgentOrchestrator(llm=langchain_llm)

orch.add_agent('analyzer', AGENT_PATH)
orch.add_agent('solver', AGENT_PATH)
orch.add_agent('verifier', AGENT_PATH)

orch.connect('analyzer', 'solver')
orch.connect('solver', 'verifier')

result = orch.run({'messages': [HumanMessage(content='Calculate 15% of 200')]})
print("Workflow result keys:", list(result.keys()))

---

## 8. Workflow Templates

Pre-built patterns for common workflow structures.

In [ ]:
from agentcomet.workflows import WorkflowBuilder, WorkflowTemplates

workflow = WorkflowTemplates.pipeline({
    'step1': AGENT_PATH,
    'step2': AGENT_PATH,
    'step3': AGENT_PATH
})
print(f"Pipeline agents: {list(workflow.agents.keys())}")

workflow2 = WorkflowTemplates.fan_out_fan_in(
    start_agent={'distributor': AGENT_PATH},
    parallel_agents={'w1': AGENT_PATH, 'w2': AGENT_PATH},
    end_agent={'aggregator': AGENT_PATH}
)
print(f"Fan-out agents: {list(workflow2.agents.keys())}")

---

## Summary — All AgentComet Features

| Feature | Method | Description |
|---------|--------|-------------|
| **Agent SDK** | `Agent`, `create_agent` | Build agents natively |
| **@tool Decorator** | `@tool` | Auto-generate ToolSpec from functions |
| **Builtin Tools** | `read`, `write` | File operations out of the box |
| **LLM Providers** | `Ollama(model=...)` | Pass directly, no `.as_langchain()` |
| **UAF Export** | `agent.export(path)` | Export to portable .uaf |
| **UAF Load** | `load_agent(path)` | Auto-route SDK-aware loading |
| Legacy Agent | `UAFAgent(name, path, llm)` | Load .uaf with LangChain LLM |
| Memory | `memory="full"` / `memory=5` | Full or sliding window |
| **Save Agent** | `agent.save_agent(path)` | Embed state in UAF |
| **Save State** | `agent.save_state()` | External versioned save |
| **Rollback** | `agent.load_state(hash)` | Switch checkpoints |
| Reload | `agent.reload()` | Hot-reload logic |
| Orchestrate | `AgentOrchestrator(llm)` | Multi-agent |
| Pipeline | `WorkflowTemplates.pipeline()` | Linear chain |